# 3. Train Dynamic Sign Model (Bi-LSTM + Multi-Head Attention)

**Kiến trúc**: 2× Bi-LSTM + Multi-Head Attention (4 heads) + Dense

**Input**: Sequence (30 frames × 252 features) = landmarks(126) + motion(126)

**Tham khảo**:
- Paper ĐH Giao thông Vận tải (TNU Journal 2025): Bi-LSTM + Multi-Head Attention đạt **99.51%**
- Motion features (frame differencing) từ AIO2025 SMIF concept

> Chạy notebook `01_data_preparation.ipynb` trước để có data.

In [ ]:
# === CẤU HÌNH ===
DATA_DIR = "../data/processed/dynamic"
OUTPUT_DIR = "../models"
EPOCHS = 20
BATCH_SIZE = 32
SEQUENCE_LENGTH = 30   # Số frames mỗi sequence (uniform sampling)
USE_MOTION = True      # Thêm motion features (frame differencing)
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 3.1 Load và tiền xử lý sequences

In [ ]:
# --- Preprocessing functions ---
def uniform_sample_frames(total_frames, target_frames, stride=1):
    """Uniform sampling + temporal padding (lặp frame cuối, không zero-pad)."""
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    if total_frames == 1:
        return np.zeros(target_frames, dtype=np.int64)
    steps = max(target_frames * stride, target_frames)
    grid = np.linspace(0, total_frames - 1, num=steps)
    idxs = grid[::stride].astype(np.int64)
    if len(idxs) < target_frames:
        pad = np.full(target_frames - len(idxs), idxs[-1], dtype=np.int64)
        idxs = np.concatenate([idxs, pad])
    return idxs[:target_frames]


def normalize_landmarks(landmarks):
    """Chuẩn hóa: wrist = gốc tọa độ, scale theo khoảng cách lớn nhất."""
    def _normalize_single(frame):
        result = frame.copy()
        for hand_offset in [0, 63]:
            hand = result[hand_offset:hand_offset + 63]
            if np.all(hand == 0):
                continue
            points = hand.reshape(21, 3)
            wrist = points[0].copy()
            points -= wrist
            max_dist = np.max(np.linalg.norm(points, axis=1))
            if max_dist > 0:
                points /= max_dist
            result[hand_offset:hand_offset + 63] = points.flatten()
        return result

    if landmarks.ndim == 1:
        return _normalize_single(landmarks)
    return np.array([_normalize_single(f) for f in landmarks])


def compute_motion_features(sequence):
    """Frame differencing: concat [position, motion]. (T,126) → (T,252)"""
    motion = np.zeros_like(sequence)
    motion[1:] = sequence[1:] - sequence[:-1]
    return np.concatenate([sequence, motion], axis=-1)


def process_sequence(seq, sequence_length, use_motion):
    """Pipeline: sample → normalize → motion features."""
    total_frames = len(seq)
    if total_frames < 3:
        return None
    feature_dim = seq.shape[1]
    if feature_dim > 126:
        seq = seq[:, :126]
    elif feature_dim < 126:
        pad = np.zeros((total_frames, 126 - feature_dim))
        seq = np.concatenate([seq, pad], axis=1)
    indices = uniform_sample_frames(total_frames, sequence_length)
    seq = seq[indices]
    seq = normalize_landmarks(seq)
    if use_motion:
        seq = compute_motion_features(seq)
    return seq


# --- Load data ---
X, y, labels = [], [], {}
class_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

for idx, class_name in enumerate(class_dirs):
    labels[idx] = class_name
    class_path = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_path):
        if not fname.endswith(".npy"):
            continue
        seq = np.load(os.path.join(class_path, fname))
        if seq.ndim == 2:
            processed = process_sequence(seq, SEQUENCE_LENGTH, USE_MOTION)
            if processed is not None:
                X.append(processed)
                y.append(idx)

X, y = np.array(X), np.array(y)
print(f"Loaded: {len(X)} sequences, {len(labels)} classes")
print(f"Input shape: {X.shape}")

In [ ]:
# Train/Val/Test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    random_state=42, stratify=y_trainval
)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 3.2 Build model Bi-LSTM + Attention

In [ ]:
class MultiHeadAttentionBlock(layers.Layer):
    """Multi-Head Attention cho temporal focusing."""
    def __init__(self, embed_dim, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.layernorm = layers.LayerNormalization()

    def call(self, x):
        attn_output = self.attention(query=x, value=x, key=x)
        return self.layernorm(x + attn_output)

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "num_heads": self.num_heads})
        return config


def build_dynamic_model(num_classes, sequence_length=30, input_dim=126, use_motion=True):
    """Bi-LSTM + Multi-Head Attention."""
    feature_dim = input_dim * 2 if use_motion else input_dim
    inputs = keras.Input(shape=(sequence_length, feature_dim), name="sequence")

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inputs)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = MultiHeadAttentionBlock(embed_dim=128, num_heads=4)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(num_classes, activation="softmax", name="output")(x)

    return keras.Model(inputs=inputs, outputs=x, name="dynamic_bilstm_attention")


num_classes = len(labels)
model = build_dynamic_model(num_classes, SEQUENCE_LENGTH, use_motion=USE_MOTION)
model.summary()

## 3.3 Training

In [ ]:
# Class weights
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}

# Compile
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.0005, weight_decay=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]

# Train!
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

## 3.4 Đánh giá kết quả

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"], label="Train")
ax1.plot(history.history["val_accuracy"], label="Validation")
ax1.set_title("Accuracy")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot(history.history["loss"], label="Train")
ax2.plot(history.history["val_loss"], label="Validation")
ax2.set_title("Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Test evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Confusion matrix
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
label_names = [labels[i].split("_", 1)[1] if "_" in labels[i] else labels[i] for i in range(num_classes)]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - Dynamic Bi-LSTM + Attention")
plt.tight_layout()
plt.show()

## 3.5 Lưu model

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model
model_path = os.path.join(OUTPUT_DIR, "dynamic_bilstm_att.keras")
model.save(model_path)
print(f"Model saved: {model_path}")

# Save labels
labels_path = os.path.join(OUTPUT_DIR, "dynamic_labels.json")
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in labels.items()}, f, ensure_ascii=False, indent=2)
print(f"Labels saved: {labels_path}")

# Save history
hist_path = os.path.join(OUTPUT_DIR, "dynamic_history.json")
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(hist_path, "w") as f:
    json.dump(hist_data, f)
print(f"History saved: {hist_path}")

print(f"\nFinal test accuracy: {test_acc:.4f}")

---
**Done!** Cả 2 models đã được train và lưu vào `../models/`. Chạy Streamlit app để deploy inference:

```bash
cd .. && streamlit run app.py
```